# AI-Driven Phishing Email Detection Using NLP

**End-to-end walkthrough**: data exploration → cleaning → feature engineering → model training → comparison → error analysis → conclusion.

This notebook does not retrain models from scratch -- it loads the already-trained artifacts produced by the scripts in `src/` (01-08), and reuses their functions directly via `importlib` where relevant. This keeps the notebook honest: it's a presentation/analysis layer over the real pipeline, not a duplicate copy of the logic living in two places at once.


## Setup

In [ ]:
import sys
import json
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy import sparse
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_RAW = PROJECT_ROOT / "data" / "raw" / "Phishing_Email.csv"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

sys.path.insert(0, str(SRC_DIR))

def load_module(filename, modname):
    """Load a src/ script (filenames start with digits, so they can't be
    imported with a normal `import` statement) as a proper module, so we can
    reuse its functions instead of copy-pasting logic into this notebook."""
    spec = importlib.util.spec_from_file_location(modname, SRC_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

print("Project root:", PROJECT_ROOT)

## 1. Data Loading & Exploration

Reusing the exact logic from `src/01_explore_data.py`.

In [ ]:
df_raw = pd.read_csv(DATA_RAW)
print("Shape:", df_raw.shape)
df_raw.head(3)

In [ ]:
print("Missing values:")
print(df_raw.isnull().sum())
print("\nClass balance:")
print(df_raw["Email Type"].value_counts())
print(df_raw["Email Type"].value_counts(normalize=True).round(3))

**Observation:** 60.7% Safe / 39.3% Phishing -- a mild imbalance, handled later via `class_weight="balanced"` (scikit-learn models) and manually-computed class weights (Neural Network).

## 2. Data Cleaning & Metadata Feature Extraction

Reusing `clean_text()` and `extract_metadata_features()` directly from `src/02_clean_data.py` via `importlib`, so this notebook demonstrates the real cleaning logic rather than a re-typed copy of it.

In [ ]:
clean_mod = load_module("02_clean_data.py", "clean_data_mod")

import nltk
from nltk.corpus import stopwords
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

sample_raw = df_raw["Email Text"].dropna().iloc[2]
sample_cleaned = clean_mod.clean_text(sample_raw, stop_words)

print("BEFORE:\n", sample_raw[:300])
print("\nAFTER:\n", sample_cleaned[:300])

In [ ]:
# Load the already-cleaned + feature-engineered dataset produced by 02_clean_data.py
df_clean = pd.read_csv(DATA_PROCESSED / "cleaned_emails.csv")
print("Cleaned shape:", df_clean.shape)
df_clean.head(3)

**Metadata features extracted:** `char_count`, `word_count`, `url_count`, `has_url`, `exclamation_count`, `digit_count`, `caps_ratio` -- pulled from the RAW text before cleaning destroys those signals (e.g. URLs are stripped during cleaning, so `url_count` has to be captured first).

## 3. Feature Engineering: TF-IDF + Metadata

Loading the saved train/test feature matrices from `src/03_feature_engineering.py` (train/test split happens *before* fitting TF-IDF/scaler, to avoid data leakage -- see that script's docstring for why this matters).

In [ ]:
X_train = sparse.load_npz(DATA_PROCESSED / "X_train.npz")
X_test = sparse.load_npz(DATA_PROCESSED / "X_test.npz")
y_train = np.load(DATA_PROCESSED / "y_train.npy")
y_test = np.load(DATA_PROCESSED / "y_test.npy")

vectorizer = joblib.load(MODELS_DIR / "tfidf_vectorizer.joblib")
feature_names = joblib.load(MODELS_DIR / "feature_names.joblib")

print("Train matrix:", X_train.shape, " Test matrix:", X_test.shape)
print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))
print("Total feature count (TF-IDF + metadata):", len(feature_names))

## 4. Model Training & Evaluation

Four classifiers were trained on this feature set (Logistic Regression, Random Forest on TF-IDF+metadata; Naive Bayes on TF-IDF only, due to its non-negativity requirement; a Keras Neural Network on the full dense matrix). Rather than retraining here, we load each saved model and reproduce its evaluation on the same held-out test set.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {}

# Logistic Regression
lr = joblib.load(MODELS_DIR / "logistic_regression.joblib")
y_pred_lr = lr.predict(X_test)
results["Logistic Regression"] = {
    "accuracy": accuracy_score(y_test, y_pred_lr),
    "precision": precision_score(y_test, y_pred_lr),
    "recall": recall_score(y_test, y_pred_lr),
    "f1": f1_score(y_test, y_pred_lr),
}

# Naive Bayes (TF-IDF only -- first 5000 columns)
nb = joblib.load(MODELS_DIR / "naive_bayes.joblib")
y_pred_nb = nb.predict(X_test[:, :5000])
results["Naive Bayes"] = {
    "accuracy": accuracy_score(y_test, y_pred_nb),
    "precision": precision_score(y_test, y_pred_nb),
    "recall": recall_score(y_test, y_pred_nb),
    "f1": f1_score(y_test, y_pred_nb),
}

# Random Forest
rf = joblib.load(MODELS_DIR / "random_forest.joblib")
y_pred_rf = rf.predict(X_test)
results["Random Forest"] = {
    "accuracy": accuracy_score(y_test, y_pred_rf),
    "precision": precision_score(y_test, y_pred_rf),
    "recall": recall_score(y_test, y_pred_rf),
    "f1": f1_score(y_test, y_pred_rf),
}

# Neural Network
from tensorflow import keras
nn = keras.models.load_model(MODELS_DIR / "neural_network.keras")
X_test_dense = X_test.toarray().astype("float32")
y_pred_nn = (nn.predict(X_test_dense, verbose=0) > 0.5).astype(int).flatten()
results["Neural Network"] = {
    "accuracy": accuracy_score(y_test, y_pred_nn),
    "precision": precision_score(y_test, y_pred_nn),
    "recall": recall_score(y_test, y_pred_nn),
    "f1": f1_score(y_test, y_pred_nn),
}

results_df = pd.DataFrame(results).T.round(4)
results_df

## 5. Model Comparison

In [ ]:
ax = results_df.plot(kind="bar", figsize=(10, 6), ylim=(0.85, 1.0))
plt.title("Model Comparison: Accuracy, Precision, Recall, F1")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Reading the comparison:**

- **Neural Network** wins on accuracy (97.18%), recall (98.15%), and F1 (96.47%) -- the strongest overall model, at the cost of being the least interpretable and the only one requiring dense arrays / GPU-friendly training.
- **Logistic Regression** is a close second (96.78% / 95.97% F1) and fully interpretable -- every prediction can be traced to specific word/metadata coefficients.
- **Random Forest** has the single highest recall (98.70%) of any model -- catches the most phishing emails, at the cost of the lowest precision (91.33%, more false alarms).
- **Naive Bayes**, using TF-IDF text alone (no metadata), lands within ~1 point of Logistic Regression -- showing how much signal phishing language carries by itself.

**Which model to deploy depends on the cost you're optimizing for:** if missing a phishing email is far more costly than a false alarm, Random Forest's recall edge may matter more than its lower overall accuracy.

## 6. Feature Importance

Logistic Regression coefficients are directly interpretable: positive = pushes toward phishing, negative = pushes toward safe.

In [ ]:
coefs = lr.coef_[0]
order = np.argsort(coefs)
top_phishing = order[-10:][::-1]
top_safe = order[:10]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh([feature_names[i] for i in top_phishing][::-1], [coefs[i] for i in top_phishing][::-1], color="crimson")
axes[0].set_title("Top features -> PHISHING")
axes[1].barh([feature_names[i] for i in top_safe][::-1], [coefs[i] for i in top_safe][::-1], color="seagreen")
axes[1].set_title("Top features -> SAFE")
plt.tight_layout()
plt.show()

**Validates the project brief's hypothesis directly:** top phishing-indicative words are `click`, `remove`, `free`, `http`, `money`, `site` -- classic urgency/sales/call-to-action language.

**Caveat:** top safe-indicative words (`enron`, `vince`, `university`, `linguistics`) reflect this dataset's specific source (Enron corporate email + academic linguistics discussion threads) rather than "legitimate email" in general -- see `reports/limitations.md` for the full discussion of why this matters for generalization.

## 7. Error Analysis

Full findings are documented in `reports/limitations.md`. Summary of the Neural Network's misclassifications on the test set:

- **78 false positives, 27 false negatives** out of 3,726 test emails.
- A subset of false positives are dataset rows with placeholder/empty body content -- a data quality artifact, not a model failure, since there's no real content to classify.
- False negatives cluster into three patterns: (1) possible label noise at the spam/phishing boundary, (2) link-farm/MLM-style spam lacking urgency language, and (3) **content obfuscation** -- messages padded with unrelated plausible-sounding text specifically to dodge keyword-based detection. This last pattern is a structural blind spot of TF-IDF-based models, since there's no suspicious vocabulary to key off -- a natural next step would be context-aware models (e.g. transformer-based embeddings) that read meaning rather than counting words.

## 8. Conclusion & Recommendations

- All four models comfortably exceed 95% accuracy on this task, confirming that phishing emails carry strong, learnable textual and structural signals.
- The **Neural Network** is the best-performing model overall and the recommended choice if raw performance is the priority.
- **Logistic Regression** is the recommended choice if interpretability matters (e.g. needing to explain *why* an email was flagged, for compliance or user trust reasons) -- it trails the NN by only ~0.4 points of accuracy.
- **Random Forest** is the recommended choice if minimizing missed phishing emails (recall) is the overriding priority, even at the cost of more false alarms.
- **Naive Bayes** demonstrates that text content alone (no metadata) already gets remarkably close to the top models -- useful if metadata features are unavailable at inference time (e.g. classifying raw pasted text with no email headers).
- **Future work:** address the content-obfuscation blind spot with context-aware embeddings; validate generalization on legitimate email sources beyond the Enron corpus; expand the dataset with the synthetic LLM-generated phishing samples identified during dataset selection, to test robustness against modern phishing styles.